# OOF(Out-Of-Fold) 임베딩 — 왜, 어떻게 (설명 + 실습)

이 노트북은 **하이브리드(GNN 임베딩 + XGBoost)에서 생기는 누수**와 그 해결책 **OOF**를 코드를 직접 돌려 보면서 이해하는 용도입니다.
(실제 5시드 본 실험은 `colab_hybrid.ipynb`. 여기서는 **시드 0 하나**만 돌려서 빠르게 개념을 확인합니다. GPU 런타임 권장, 약 10분.)

| 단계 | 내용 |
|---|---|
| 0 | 준비 (저장소, 데이터, 설치) |
| 1 | 데이터 구조 보기: 시간 단계와 라벨 |
| 2 | **누수가 뭔지** 눈으로 확인: 정답을 본 노드 vs 못 본 노드 |
| 3 | OOF 폴드 나누기 |
| 4 | **폴드 하나**를 손으로 해보기 (GNN 학습 → 임베딩 추출) |
| 5 | 좌표계 맞추기 (Procrustes 정렬) |
| 6 | 5개 폴드 전부 → OOF 임베딩 완성 |
| 7 | 하이브리드 XGBoost로 비교: 원본만 / 누수 있음 / OOF |

## 0. 준비

In [ ]:
import getpass, os
if not os.path.exists('2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone -q --single-branch -b main https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git
%cd 2026-2-HAI-ProjectTeam8
!git pull -q origin main || true
!git fetch -q origin gcn:refs/remotes/origin/gcn
!ls hybrid

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/elliptic
!cp "/content/drive/MyDrive/Colab Notebooks/elliptic/"*.csv data/elliptic/
!pip install -q torch-geometric "xgboost>=3.4.1,<3.5" scipy
# 튜닝 후 GCN으로 저장해 둔 '기준 임베딩'(= Train+Val 정답을 모두 본 GNN의 임베딩) 시드 0
!mkdir -p results/emb
!git show origin/gcn:results/emb/emb_gcn_seed0.npy > results/emb/emb_gcn_seed0.npy
!git show origin/gcn:results/metrics/gcn.csv > gcn_metrics.csv

## 1. 데이터 구조: 시간 단계와 라벨
Elliptic은 비트코인 거래 20만 개가 49개 시간 단계에 걸쳐 있고, 일부만 정답(1=불법, 0=합법)이 있고 나머지는 `y=2`(라벨 없음)입니다.
분할: **학습 1~29, 검증 30~34, 테스트 35~49**.

In [ ]:
import os
import numpy as np, pandas as pd, torch
from sklearn.metrics import average_precision_score as AP
from common.data import load_data, to_pyg, VAL_END
from common.train_gnn import fit, predict, class_weight
from common.utils import set_seed
from hybrid import make_oof_emb as oof

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = load_data('data/elliptic', standardize=True)
g = to_pyg(data).to(dev); g.x, g.y = g.x.float(), g.y.long()
y, step = data.y, data.time_step
print('장치', dev, '| 노드', data.num_nodes)
for name, m in [('학습 1~29', step<=29), ('검증 30~34', (step>=30)&(step<=34)), ('테스트 35~49', step>=35)]:
    print(f'{name}: 노드 {m.sum():6d} | 불법 {(m&(y==1)).sum():5d} 합법 {(m&(y==0)).sum():6d} 라벨없음 {(m&(y==2)).sum():6d}')

## 2. 누수가 뭔지 눈으로 보기
GNN은 **학습에 쓴 노드의 정답은 잘 맞히고(외운 것), 처음 보는 노드는 덜 맞힙니다.**
저장된 임베딩을 만든 GNN은 1~34의 정답을 모두 보고 학습했습니다 → 1~34 노드의 임베딩에는 이미 그 노드의 정답이 스며 있습니다.
그런데 XGBoost가 그 임베딩을 학습 입력으로 쓰면 "정답이 적힌 힌트"로 공부하게 되고, 정답을 못 본 테스트 노드에서는 그 힌트가 없어서 무너집니다.

아래는 **같은 GNN 구조를 두 가지로 학습**해서 "자기 정답을 본 노드"와 "못 본 노드"의 점수를 비교합니다 (폴드 1 = 시간 단계 1~7을 대상으로).

In [ ]:
folds = oof.time_folds(5)
print('폴드:', [f'{f[0]}~{f[-1]}' for f in folds])

import argparse
from models.gcn import GCN
args = argparse.Namespace(hidden=64, dropout=0.5, lr=0.01, weight_decay=5e-4)
EPOCHS = 144            # 시드 0의 best_epoch (튜닝 후 GCN)
factory = lambda d, a: GCN(d, a.hidden, a.dropout)

held   = np.isin(step, folds[0])                      # 폴드 1: 시간 단계 1~7의 모든 노드
labeled = (y != 2) & (step <= VAL_END)
held_lab = held & labeled

def train_gnn(train_mask):
    set_seed(0)
    model = factory(g.x.shape[1], args).to(dev)
    fit(model, g, torch.from_numpy(train_mask).to(dev), class_weight(y[train_mask], dev), args.lr, args.weight_decay, EPOCHS)
    return predict(model, g)          # (확률, 임베딩)

# (가) 누수 있는 방식: 1~34 정답 전부로 학습 (폴드 1의 정답도 봄)
prob_leak, emb_leak = train_gnn(labeled)
# (나) OOF 방식: 폴드 1을 뺀 나머지 정답으로만 학습 (폴드 1의 정답은 본 적 없음)
train_k = labeled & ~held
prob_oof, emb_oof = train_gnn(train_k)

print(f'폴드 1(1~7) 라벨 노드 {held_lab.sum()}개, 그 중 불법 {(held_lab&(y==1)).sum()}개')
print(f'(가) 폴드 1 정답을 본 GNN  → 폴드 1 PR-AUC: {AP(y[held_lab], prob_leak[held_lab]):.3f}   ← 외운 점수, 부풀려짐')
print(f'(나) 폴드 1 정답을 못 본 GNN → 폴드 1 PR-AUC: {AP(y[held_lab], prob_oof[held_lab]):.3f}   ← 진짜 실력')
print(f'참고 (나)가 학습에 쓴 노드들의 PR-AUC: {AP(y[train_k], prob_oof[train_k]):.3f}')

(가)와 (나)의 차이가 바로 **누수의 크기**입니다. 하이브리드의 XGBoost는 1~34 노드에 대해 (가)처럼 부풀려진 임베딩을 보고 공부하고,
테스트(35~)에서는 (나)처럼 정직한 임베딩만 받습니다. 학습과 테스트의 입력 분포가 달라지는 것이 문제입니다.

**OOF의 목표:** 1~34의 모든 노드에 대해 **"자기 정답을 못 본 GNN이 만든 임베딩"**을 채워 넣자. 그러면 학습·테스트 입력이 같은 성격이 됩니다.

## 3~4. 폴드 하나를 손으로: 학습 → 임베딩 추출
위 (나)가 바로 폴드 1의 OOF 단계입니다. 규칙은 이렇습니다.
1. 시간 단계 1~34를 **시간 순서대로** 연속된 5개 블록으로 나눈다 (랜덤 분할 X. 같은 시간대 노드끼리 연결돼 있어서 섞으면 이웃을 통해 새기 때문).
2. 블록 k의 정답은 전부 숨기고, 나머지 블록의 정답으로 GNN을 학습한다.
3. 학습된 GNN으로 **전체 그래프를 forward** → 블록 k 노드의 임베딩만 가져온다. (피처와 간선은 써도 되지만 **블록 k의 정답은 쓰지 않았음**)
4. 이걸 k=1..5로 반복해서 임베딩 표를 채운다.

에폭 수는 검증 구간을 따로 쓸 수 없으니 **본 학습의 best_epoch로 고정**합니다 (시드 0 → 144).

## 5. 좌표계 맞추기 (Procrustes 정렬)
폴드마다 GNN을 새로 학습하므로, 같은 노드라도 64개 숫자의 **좌표축이 서로 달라집니다** (축이 돌아가 있거나 뒤집혀 있음).
그대로 이어 붙이면 XGBoost 입력에서 "3번째 열"의 의미가 폴드마다 제각각이 됩니다.
그래서 블록 k의 **라벨 없는 노드(y=2)** 에서 "폴드 임베딩 → 기준 임베딩"으로 가는 회전/반사 + 크기 + 이동을 구해서 블록 k 전체에 적용합니다.
정렬에는 정답을 전혀 쓰지 않으므로 누수가 생기지 않습니다.

In [ ]:
unl = held & (y == 2)
ref = np.load('results/emb/emb_gcn_seed0.npy')
f = oof.procrustes_fit(emb_oof[unl], ref[unl])
al = oof.procrustes_apply(emb_oof, f)

print(f'정렬 전 코사인 유사도(폴드 임베딩 vs 기준): {oof.mean_cosine(emb_oof[unl], ref[unl]):+.3f}   (≈0이면 좌표축이 서로 무관)')
print(f'정렬 후  (라벨 없는 노드, 정렬에 사용):       {oof.mean_cosine(al[unl], ref[unl]):+.3f}')
print(f'정렬 후  (라벨 있는 노드, 정렬에 안 쓰임):    {oof.mean_cosine(al[held_lab], ref[held_lab]):+.3f}   ← 이게 높아야 정렬이 일반화된 것')

라벨 있는 노드는 정렬에 쓰이지 않았는데도 코사인이 높다면, 정렬이 "라벨에 맞춘 게 아니라 좌표축 자체를 맞춘 것"이라는 증거입니다.
참고로 기준 임베딩은 정답을 본 모델이라 라벨 노드에서는 (가)처럼 조금 더 '또렷'합니다. 그래서 정렬 후 코사인이 1.0까지는 오르지 않는 게 정상입니다.

## 6. 5개 폴드 전부 → OOF 임베딩 완성 (시드 0)
위의 과정을 5번 돌리는 함수가 `hybrid.make_oof_emb.oof_embeddings`입니다. 테스트 구간(35~49)은 정답을 본 적이 없으니 기준 임베딩을 그대로 씁니다.

In [ ]:
emb_final, rows = oof.oof_embeddings(factory, args, data, g, ref, seed=0, epochs=EPOCHS, device=dev)
rep = pd.DataFrame(rows)
print(rep[['fold','steps','n_train','cos_raw','cos_aligned_unlabeled','cos_aligned_labeled','oof_pr_auc']].round(3).to_string(index=False))
print('\n폴드 모델들의 OOF PR-AUC(1~34 전체, 각자 못 본 노드만):', round(rep.oof_pr_auc_pooled.iloc[0], 4))

te = step > VAL_END
print('테스트 노드 임베딩이 기준과 동일:', bool(np.array_equal(emb_final[te], ref[te])))
print('학습/검증 노드 임베딩이 기준과 다름:', bool(not np.allclose(emb_final[~te], ref[~te], atol=1e-3)))
os.makedirs('oof_demo/emb_oof', exist_ok=True)
np.save('oof_demo/emb_oof/emb_oof_gcn_seed0.npy', emb_final.astype(np.float32))

## 7. 하이브리드로 비교: 원본만 / 누수 있음 / OOF
XGBoost 설정은 모두 기준선의 최적값으로 고정하고, **입력만** 바꿉니다 (시드 0만).
- `xgb_check`: 원본 165개
- `hybrid_gcn_naive`: 원본 + **기준 임베딩**(누수 있음)
- `hybrid_gcn`: 원본 + **OOF 임베딩**

In [ ]:
R = 'oof_demo/res'
!python -m hybrid.skip_gcn --mode raw --seeds 0 --results-dir {R} > /dev/null
!python -m hybrid.skip_gcn --emb-models gcn --seeds 0 --results-dir {R} > /dev/null
!python -m hybrid.skip_gcn --emb-models gcn --emb-kind oof --emb-dir oof_demo/emb_oof --seeds 0 --results-dir {R} > /dev/null

def get(name, split='test', tt='fixed_0.5'):
    m = pd.read_csv(f'{R}/metrics/{name}.csv', dtype={'threshold_type': str})
    return m[(m.split==split)&(m.threshold_type==tt)].iloc[0]
rows = []
for name, label in [('xgb_check','원본만'), ('hybrid_gcn_naive','누수 있음'), ('hybrid_gcn','OOF')]:
    t, v, tr = get(name), get(name,'val','val_opt'), get(name,'train','fixed_0.5')
    rows.append(dict(입력=label, train_PRAUC=tr.pr_auc, val_PRAUC=v.pr_auc, test_PRAUC=t.pr_auc, test_F1=t.f1))
res = pd.DataFrame(rows).round(4); res['train-test 차이'] = (res.train_PRAUC - res.test_PRAUC).round(4)
print(res.to_string(index=False))

### 읽는 법
- **누수 있음**: train/val 점수는 높은데(임베딩에 정답이 있으니까) **test는 오히려 떨어집니다** → 학습-테스트 간극이 가장 큼. 이게 누수의 증상입니다.
- **OOF**: 간극이 줄고 test가 원본만 쓴 기준선 수준으로 회복됩니다. 그 이상(눈에 띄는 향상)은 이 데이터에서 아직 확실하지 않습니다.
- 시드 하나의 숫자라서 ±0.003 정도 차이는 우연일 수 있습니다. 보고서 숫자는 5시드 평균·`colab_hybrid.ipynb`로 정합니다.

### 한 줄 정리
> OOF = "각 노드의 임베딩을, 그 노드의 정답을 못 본 모델에게서 받는다". 시간 블록으로 나눠 돌아가며 한 블록씩 정답을 숨기고, 좌표계는 라벨 없는 노드로 맞춘다.
> (Kaggle stacking의 OOF 예측과 같은 원리. 여기서는 '예측'이 아니라 '임베딩'을 OOF로 만든 것.)

### 전체 5시드를 CLI로 돌릴 때
```
python -m hybrid.make_oof_emb --model gcn --lr 0.01 --dropout 0.5 --weight-decay 5e-4 --metrics-csv gcn_metrics.csv
python -m hybrid.skip_gcn --emb-models gcn --emb-kind oof
```